## STEP 1 : Sentence Embedding Generation

In [1]:
# Phase 1 - Offline Retrieval System
# Step 1: Sentence Embedding Generation

# Import libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import pickle


# Define paths
data_path = Path("/Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/merged_factcheck_datasetcleaned.csv")
save_dir = Path("/Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/embeddings")
# Create directory if not exists
save_dir.mkdir(parents=True, exist_ok=True)


# Loading Final Cleaned dataset
df = pd.read_csv(data_path)
print(f"✅ Loaded dataset with shape: {df.shape}")
print(df.head(20))

# Ensure required columns exist
required_cols = ['claim_id', 'claim_text', 'verdict_mapped', 'summary', 'url', 'dataset_source']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")



# Load Sentence-BERT model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
print(f"✅ Loaded embedding model: {model_name}")

# Generate embeddings

# Convert claim_text to list
texts = df['claim_text'].astype(str).tolist()

print("⚙️ Generating embeddings... this may take a few minutes...")
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # Normalize for cosine similarity
)

print(f"✅ Embedding generation complete. Shape: {embeddings.shape}")


# Create metadata for FAISS
metadata = df[['claim_id', 'claim_text', 'verdict_mapped', 'summary', 'url', 'dataset_source']].copy()
print(f"✅ Metadata prepared with shape: {metadata.shape}")


# Save embeddings & metadata
embeddings_file = save_dir / "claim_embeddings.npy"
metadata_file = save_dir / "claim_metadata.csv"
pickle_file = save_dir / "claim_embeddings.pkl"

# Save NumPy array
np.save(embeddings_file, embeddings)
print(f" Saved embeddings array to: {embeddings_file}")

# Save metadata CSV
metadata.to_csv(metadata_file, index=False)
print(f" Saved metadata CSV to: {metadata_file}")

# Optional: save combined pickle (for faster loading in FAISS pipeline)
with open(pickle_file, "wb") as f:
    pickle.dump({"embeddings": embeddings, "metadata": metadata}, f)
print(f"Saved combined pickle file to: {pickle_file}")



print("Sample metadata entry:")
print(metadata.head(10))
print(f"Embeddings dtype: {embeddings.dtype}, shape: {embeddings.shape}")

✅ Loaded dataset with shape: (25540, 9)
   claim_id                                         claim_text  \
0    P00001  John McCain opposed bankruptcy protections for...   
1    P00002  "Bennie Thompson actively cheer-led riots in t...   
2    P00003  Says Maggie Hassan was "out of state on 30 day...   
3    P00004  "BUSTED: CDC Inflated COVID Numbers, Accused o...   
4    P00005  "I'm the only (Republican) candidate that has ...   
5    P00006  "There are actually only 30 countries that pra...   
6    P00007  "My husband and I have never gotten a penny of...   
7    P00008  "If you go strictly by the numbers, crime is d...   
8    P00009  "The American people say, don't touch Social S...   
9    P00010  "Since 1978, CEO compensation rose over 1,000%...   
10   P00011  Says her accomplishments include "a fiscally r...   
11   P00012  Says President Obama’s deal "allows Iran to pr...   
12   P00013  Says "Donald Trump says climate change is a ho...   
13   P00014  "At least 450,000 ballo

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
⚙️ Generating embeddings... this may take a few minutes...


Batches:   0%|          | 0/400 [00:00<?, ?it/s]

✅ Embedding generation complete. Shape: (25540, 384)
✅ Metadata prepared with shape: (25540, 6)
 Saved embeddings array to: /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/embeddings/claim_embeddings.npy
 Saved metadata CSV to: /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/embeddings/claim_metadata.csv
Saved combined pickle file to: /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/embeddings/claim_embeddings.pkl
Sample metadata entry:
  claim_id                                         claim_text verdict_mapped  \
0   P00001  John McCain opposed bankruptcy protections for...    Likely True   
1   P00002  "Bennie Thompson actively cheer-led riots in t...   Likely False   
2   P00003  Says Maggie Hassan was "out of state on 30 day...    Likely True   
3   P00004  "BUSTED: CDC Inflated COVID Numbers, Accused o...   Likely False   
4   P00005  "I'm the only (Republican) candidate that has ...      Uncertain

## STEP 2 : Build FAISS Index

In [1]:
# Import libraries
import os
import faiss
import numpy as np
import pandas as pd
from pathlib import Path
import pickle


# Define paths
base_path = Path("/Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject")
embed_dir = base_path / "data/processed/embeddings"
save_dir = base_path / "data/processed/faiss_index"

# Create directory if not exists
save_dir.mkdir(parents=True, exist_ok=True)
embeddings_file = embed_dir / "claim_embeddings.npy"
metadata_file = embed_dir / "claim_metadata.csv"


# Load embeddings and metadata
print("📂 Loading embeddings and metadata...")
embeddings = np.load(embeddings_file)
metadata = pd.read_csv(metadata_file)
print(f" Embeddings shape: {embeddings.shape}")
print(f" Metadata shape: {metadata.shape}")


# Normalize embeddings (for cosine similarity)
# Ensure embeddings are normalized (unit length)
faiss.normalize_L2(embeddings)


# Build FAISS index (Cosine Similarity) : We are using IndexFlatIP (Inner Product) which works with normalized embeddings as cosine similarity
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
print("⚙️ Adding embeddings to FAISS index...")
index.add(embeddings)
print(f" Added {index.ntotal} vectors to index.")

# Save FAISS index
index_file = save_dir / "claimverify_faiss_index.bin"
faiss.write_index(index, str(index_file))
print(f" FAISS index saved to: {index_file}")

# Save metadata mapping (for retrieval)
metadata_file_out = save_dir / "claimverify_faiss_metadata.csv"
metadata.to_csv(metadata_file_out, index=False)
print(f" Metadata mapping saved to: {metadata_file_out}")

# Summary
print("\n✅ FAISS Index Build Complete!")
print(f"Index file: {index_file}")
print(f"Metadata:   {metadata_file_out}")

📂 Loading embeddings and metadata...
 Embeddings shape: (25540, 384)
 Metadata shape: (25540, 6)
⚙️ Adding embeddings to FAISS index...
 Added 25540 vectors to index.
 FAISS index saved to: /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/faiss_index/claimverify_faiss_index.bin
 Metadata mapping saved to: /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/faiss_index/claimverify_faiss_metadata.csv

✅ FAISS Index Build Complete!
Index file: /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/faiss_index/claimverify_faiss_index.bin
Metadata:   /Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject/data/processed/faiss_index/claimverify_faiss_metadata.csv


## STEP 3 : Retrieval Function

In [3]:
#Sample Retrieval Function Working Example (Function code in UF-EEE6778-Fall25-TermProject/src/retrieval.py)
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.retrieval import ClaimRetrievalEngine

engine = ClaimRetrievalEngine()
results = engine.retrieve_similar_claims("COVID-19 vaccines cause infertility", top_k=5)
display(results)

📂 Loading FAISS index and metadata...
✅ Loaded FAISS index with 25540 vectors.
 Metadata records: 25540
⚙️ Loading SentenceTransformer model...
✅ Model loaded successfully.


,rank,claim_id,claim_text,similarity,verdict_mapped,summary,url,dataset_source
0,1,P12748,University of Miami researchers have found tha...,0.773241,Likely False,NaN,https://www.politifact.com/factchecks/2021/may...,PolitiFact
1,2,P20071,Women’s menstrual cycles and fertility are aff...,0.767765,Likely False,NaN,https://www.politifact.com/factchecks/2021/apr...,PolitiFact
2,3,P00936,The COVID-19 vaccines cause AIDS.,0.707184,Likely False,NaN,https://www.politifact.com/factchecks/2021/dec...,PolitiFact
3,4,P09341,"A study found an ""82% miscarriage rate"" among ...",0.682868,Likely False,NaN,https://www.politifact.com/factchecks/2021/jul...,PolitiFact
4,5,P17420,Says the surge in COVID-19 cases is caused by ...,0.654075,Likely False,NaN,https://www.politifact.com/factchecks/2021/aug...,PolitiFact
